# Exploratory Data Analysis — Predict Future Sales

**Objetivo:** entender el comportamiento de las ventas diarias de 1C Company, una de las mayores cadenas
rusas de software, videojuegos y entretenimiento, para preparar el *forecasting* de las ventas mensuales
por par tienda-producto de **noviembre de 2015**.

## Configuración inicial

### Librerías

Importación de librerías agrupadas por categoría y definición del estilo global de las gráficas.

In [1]:
# Manipulación de datos
from pathlib import Path

import numpy as np
import pandas as pd

# Gráficas
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Notebooks
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

### Parámetros y carga de datos

Rutas parametrizadas de los archivos de entrada y función de carga que valida que cada archivo exista.

In [2]:
DATA_PATH = Path("../data/raw")

FILES = {
    "sales_train": "sales_train.csv",
    "items": "items.csv",
    "item_categories": "item_categories.csv",
    "shops": "shops.csv",
    "test": "test.csv",
}


def load_csv(file_name: str, data_path: Path = DATA_PATH) -> pd.DataFrame:
    """Carga un CSV desde data_path; lanza FileNotFoundError si no existe."""
    file_path = Path(data_path) / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {file_path}")
    return pd.read_csv(file_path)

## 2. Comprensión inicial del dataset

### 2.a Carga de los archivos y vista preliminar

Se cargan los cinco archivos del dataset y se muestran sus primeras filas para reconocer su estructura.

In [3]:
datasets = {name: load_csv(file_name) for name, file_name in FILES.items()}

sales_train = datasets["sales_train"]
items = datasets["items"]
item_categories = datasets["item_categories"]
shops = datasets["shops"]
test = datasets["test"]

for name, df in datasets.items():
    print(f"{name}.csv")
    display(df.head())

sales_train.csv


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.00
1,03.01.2013,0,25,2552,899.00,1.00
2,05.01.2013,0,25,2552,899.00,-1.00
3,06.01.2013,0,25,2554,"1,709.05",1.00
4,15.01.2013,0,25,2555,"1,099.00",1.00


items.csv


,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


item_categories.csv


,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


shops.csv


,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


test.csv


,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


**Interpretación**

- `sales_train` es la tabla de hechos: cada fila es la venta de un producto (`item_id`) en una tienda
  (`shop_id`) en un día (`date`), con su precio y unidades vendidas. `date_block_num` indexa el mes.
- `items`, `item_categories` y `shops` son catálogos que traducen los identificadores a nombres. Los nombres
  están en ruso, por lo que el análisis se apoya en los identificadores.
- `items` enlaza cada producto con su categoría mediante `item_category_id`, lo que permite llevar la
  categoría a las ventas.
- `test` no tiene fecha ni ventas: solo `ID`, `shop_id` e `item_id`. Son los pares tienda-producto cuyas
  ventas de noviembre de 2015 hay que pronosticar.

### 2.b Resumen de dimensiones y memoria

Número de filas, columnas y uso de memoria (en MB, incluyendo el contenido de las columnas de texto) de cada
archivo.

In [4]:
summary = pd.DataFrame(
    [
        {
            "Archivo": FILES[name],
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Memoria (MB)": df.memory_usage(deep=True).sum() / 1024**2,
        }
        for name, df in datasets.items()
    ]
)

summary.style.format({"Filas": "{:,}", "Memoria (MB)": "{:,.2f}"}).hide(axis="index")

Archivo,Filas,Columnas,Memoria (MB)
sales_train.csv,"2,935,849",6,299.58
items.csv,"22,170",3,3.66
item_categories.csv,84,2,0.01
shops.csv,60,2,0.01
test.csv,"214,200",3,4.90


**Interpretación**

- `sales_train.csv` concentra casi todo el volumen: 2,935,849 registros diarios y 299.58 MB en memoria, frente
  a menos de 5 MB de cada uno de los demás archivos.
- Los catálogos son pequeños: 22,170 productos, 84 categorías y 60 tiendas.
- `test.csv` tiene 214,200 pares tienda-producto a pronosticar.
- El dataset cabe en memoria sin problema, aunque conviene optimizar tipos de `sales_train` antes de agregar
  o cruzar tablas.

### 2.c Diccionario de variables

Descripción de cada variable, el archivo en el que aparece y su tipo.

In [5]:
data_dictionary = pd.DataFrame(
    [
        ("date", "sales_train", "fecha", "Fecha de la venta, en formato dd.mm.yyyy."),
        ("date_block_num", "sales_train", "fecha",
         "Número consecutivo del mes: 0 = enero de 2013, 33 = octubre de 2015."),
        ("shop_id", "sales_train, shops, test", "identificador", "Identificador único de la tienda."),
        ("item_id", "sales_train, items, test", "identificador", "Identificador único del producto."),
        ("item_price", "sales_train", "numérica", "Precio unitario del producto en la venta (rublos)."),
        ("item_cnt_day", "sales_train", "numérica",
         "Unidades vendidas en el día; los valores negativos corresponden a devoluciones."),
        ("item_name", "items", "categórica", "Nombre del producto."),
        ("item_category_id", "items, item_categories", "identificador",
         "Identificador único de la categoría del producto."),
        ("item_category_name", "item_categories", "categórica", "Nombre de la categoría del producto."),
        ("shop_name", "shops", "categórica", "Nombre de la tienda."),
    ],
    columns=["Columna", "Archivo", "Tipo", "Descripción"],
)

data_dictionary.style.hide(axis="index")

Columna,Archivo,Tipo,Descripción
date,sales_train,fecha,"Fecha de la venta, en formato dd.mm.yyyy."
date_block_num,sales_train,fecha,"Número consecutivo del mes: 0 = enero de 2013, 33 = octubre de 2015."
shop_id,"sales_train, shops, test",identificador,Identificador único de la tienda.
item_id,"sales_train, items, test",identificador,Identificador único del producto.
item_price,sales_train,numérica,Precio unitario del producto en la venta (rublos).
item_cnt_day,sales_train,numérica,Unidades vendidas en el día; los valores negativos corresponden a devoluciones.
item_name,items,categórica,Nombre del producto.
item_category_id,"items, item_categories",identificador,Identificador único de la categoría del producto.
item_category_name,item_categories,categórica,Nombre de la categoría del producto.
shop_name,shops,categórica,Nombre de la tienda.


**Variable objetivo**

La competencia no pide predecir ventas diarias, sino **`item_cnt_month`**: el total de unidades vendidas de
cada producto en cada tienda durante un mes. Se obtiene agregando `item_cnt_day` por `date_block_num`,
`shop_id` e `item_id`:

$$\text{item\_cnt\_month}_{m,\,s,\,i} = \sum_{d \in m} \text{item\_cnt\_day}_{d,\,s,\,i}$$

El pronóstico corresponde al mes siguiente al último del histórico (noviembre de 2015, `date_block_num` = 34)
para cada par tienda-producto de `test.csv`.

### 2.d Rango temporal

Se convierte `date` desde el formato dd.mm.yyyy a tipo fecha y se calcula el periodo que cubre el
histórico de ventas.

In [6]:
sales_train["date"] = pd.to_datetime(sales_train["date"], format="%d.%m.%Y")

date_min = sales_train["date"].min()
date_max = sales_train["date"].max()

time_range = pd.DataFrame(
    {
        "Métrica": [
            "Fecha mínima",
            "Fecha máxima",
            "Días del periodo",
            "Días con registros",
            "Meses cubiertos (date_block_num)",
        ],
        "Valor": [
            date_min.strftime("%d/%m/%Y"),
            date_max.strftime("%d/%m/%Y"),
            f"{(date_max - date_min).days + 1:,}",
            f"{sales_train['date'].nunique():,}",
            f"{sales_train['date_block_num'].nunique()} "
            f"({sales_train['date_block_num'].min()} a {sales_train['date_block_num'].max()})",
        ],
    }
)

time_range.style.hide(axis="index")

Métrica,Valor
Fecha mínima,01/01/2013
Fecha máxima,31/10/2015
Días del periodo,"1,034"
Días con registros,"1,034"
Meses cubiertos (date_block_num),34 (0 a 33)


**Interpretación**

- El histórico va del 01/01/2013 al 31/10/2015: 1,034 días, y todos tienen al menos un registro, así que
  no hay huecos en la serie diaria.
- Esos días se agrupan en 34 meses (`date_block_num` de 0 a 33). Noviembre de 2015, el mes a pronosticar,
  corresponde a `date_block_num` = 34.
- El histórico incluye dos meses de noviembre completos (2013 y 2014), que sirven de referencia para la
  estacionalidad del mes a predecir.